In [3]:
# Customer Engagement & Product Utilization Analytics

## Initial Data Inspection

In [4]:
import pandas as pd

In [5]:
df = pd.read_csv("C:/Users/dharw/OneDrive/Documents/PROJECTS/unified mentor/Customer_Engagement_Product_Utilization_Analytics/Data/European_Bank.csv")

In [6]:
df.head()

,Year,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,2025,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2025,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,2025,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,2025,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,2025,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [7]:
df.shape

(10000, 14)

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Year             10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  object 
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  object 
 5   Gender           10000 non-null  object 
 6   Age              10000 non-null  int64  
 7   Tenure           10000 non-null  int64  
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(2), int64(9), object(3)
memory usage: 1.1+ MB


In [9]:
df.columns

Index(['Year', 'CustomerId', 'Surname', 'CreditScore', 'Geography', 'Gender',
       'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard',
       'IsActiveMember', 'EstimatedSalary', 'Exited'],
      dtype='object')

In [10]:
df.isnull().sum()

Year               0
CustomerId         0
Surname            0
CreditScore        0
Geography          0
Gender             0
Age                0
Tenure             0
Balance            0
NumOfProducts      0
HasCrCard          0
IsActiveMember     0
EstimatedSalary    0
Exited             0
dtype: int64

In [11]:
# Validate the important binary & categorical fields

print("Geography:", df["Geography"].unique())
print("Gender:", df["Gender"].unique())

print("\nHasCrCard:", df["HasCrCard"].unique())
print("IsActiveMember:", df["IsActiveMember"].unique())
print("Exited:", df["Exited"].unique())

Geography: ['France' 'Spain' 'Germany']
Gender: ['Female' 'Male']

HasCrCard: [1 0]
IsActiveMember: [1 0]
Exited: [1 0]


In [12]:
df[["CreditScore", "Age", "Tenure", "Balance",
    "NumOfProducts", "EstimatedSalary"]].describe()

,CreditScore,Age,Tenure,Balance,NumOfProducts,EstimatedSalary
count,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000
mean,650.528800,38.921800,5.012800,76485.889288,1.530200,100090.239881
std,96.653299,10.487806,2.892174,62397.405202,0.581654,57510.492818
min,350.000000,18.000000,0.000000,0.000000,1.000000,11.580000
25%,584.000000,32.000000,3.000000,0.000000,1.000000,51002.110000
50%,652.000000,37.000000,5.000000,97198.540000,1.000000,100193.915000
75%,718.000000,44.000000,7.000000,127644.240000,2.000000,149388.247500
max,850.000000,92.000000,10.000000,250898.090000,4.000000,199992.480000


In [13]:
# Create the Engagement Profiles

df["Balance"].median()

97198.54000000001

In [14]:
df["EngagementProfile"] = "Other"

df.loc[
    (df["IsActiveMember"] == 1) &
    (df["NumOfProducts"] >= 2),
    "EngagementProfile"
] = "Active Engaged"

df.loc[
    (df["IsActiveMember"] == 0) &
    (df["NumOfProducts"] == 1),
    "EngagementProfile"
] = "Inactive Disengaged"

df.loc[
    (df["IsActiveMember"] == 1) &
    (df["NumOfProducts"] == 1),
    "EngagementProfile"
] = "Active Low-Product"

df.loc[
    (df["IsActiveMember"] == 0) &
    (df["Balance"] > 97198.54),
    "EngagementProfile"
] = "Inactive High-Balance"

In [15]:
df["EngagementProfile"].value_counts()

EngagementProfile
Active Engaged           2588
Active Low-Product       2563
Inactive High-Balance    2456
Other                    1515
Inactive Disengaged       878
Name: count, dtype: int64

In [16]:
df.groupby("EngagementProfile")["Exited"].mean().sort_values(ascending=False)

EngagementProfile
Inactive Disengaged      0.398633
Inactive High-Balance    0.323290
Active Low-Product       0.189231
Other                    0.104290
Active Engaged           0.096600
Name: Exited, dtype: float64

In [17]:
engagement_churn = (
    df.groupby("IsActiveMember")["Exited"]
      .agg(["count", "sum", "mean"])
      .reset_index()
)

engagement_churn["ChurnRate"] = engagement_churn["mean"] * 100

engagement_churn

,IsActiveMember,count,sum,mean,ChurnRate
0,0,4849,1302,0.268509,26.850897
1,1,5151,735,0.142691,14.269074


In [18]:
df.groupby("IsActiveMember")["Exited"].mean() * 100

IsActiveMember
0    26.850897
1    14.269074
Name: Exited, dtype: float64

In [19]:
# Analyze Product Utilization vs Churn

product_churn = (
    df.groupby("NumOfProducts")["Exited"]
      .agg(["count", "sum", "mean"])
      .reset_index()
)

product_churn["ChurnRate"] = product_churn["mean"] * 100

product_churn

,NumOfProducts,count,sum,mean,ChurnRate
0,1,5084,1409,0.277144,27.714398
1,2,4590,348,0.075817,7.581699
2,3,266,220,0.827068,82.706767
3,4,60,60,1.000000,100.000000


In [20]:
df.groupby("NumOfProducts")["Exited"].mean() * 100

NumOfProducts
1     27.714398
2      7.581699
3     82.706767
4    100.000000
Name: Exited, dtype: float64

In [21]:
# Single-product vs multi-product

df["ProductGroup"] = df["NumOfProducts"].apply(
    lambda x: "Single-Product" if x == 1 else "Multi-Product"
)

In [22]:
product_group_churn = (
    df.groupby("ProductGroup")["Exited"]
      .agg(["count", "sum", "mean"])
      .reset_index()
)

product_group_churn["ChurnRate"] = product_group_churn["mean"] * 100

product_group_churn

,ProductGroup,count,sum,mean,ChurnRate
0,Multi-Product,4916,628,0.127746,12.774614
1,Single-Product,5084,1409,0.277144,27.714398


In [23]:
df.groupby("ProductGroup")["Exited"].mean() * 100

ProductGroup
Multi-Product     12.774614
Single-Product    27.714398
Name: Exited, dtype: float64

In [24]:
# Investigate the 3 & 4 Product Customers

df.groupby("NumOfProducts").agg(
    Customers=("CustomerId", "count"),
    ChurnRate=("Exited", "mean"),
    AvgBalance=("Balance", "mean"),
    AvgAge=("Age", "mean"),
    AvgCreditScore=("CreditScore", "mean"),
    ActiveRate=("IsActiveMember", "mean")
).reset_index()

,NumOfProducts,Customers,ChurnRate,AvgBalance,AvgAge,AvgCreditScore,ActiveRate
0,1,5084,0.277144,98551.870614,39.673092,649.120968,0.504131
1,2,4590,0.075817,51879.145813,37.753595,652.188671,0.532898
2,3,266,0.827068,75458.328195,43.195489,648.105263,0.424812
3,4,60,1.000000,93733.135000,45.683333,653.583333,0.483333


In [25]:
pd.crosstab(
    df["NumOfProducts"],
    df["IsActiveMember"],
    normalize="index"
) * 100

IsActiveMember,0,1
NumOfProducts,,
1,49.586939,50.413061
2,46.710240,53.289760
3,57.518797,42.481203
4,51.666667,48.333333


In [26]:
# Calculate the Product Depth Index

product_depth = (
    df.groupby("Exited")["NumOfProducts"]
      .agg(["count", "mean"])
      .reset_index()
)

product_depth

,Exited,count,mean
0,0,7963,1.544267
1,1,2037,1.475209


In [27]:
df.groupby("NumOfProducts")["Exited"].agg(
    Customers="count",
    Churned="sum"
).reset_index()

,NumOfProducts,Customers,Churned
0,1,5084,1409
1,2,4590,348
2,3,266,220
3,4,60,60


In [28]:
# Balance vs Engagement

balance_engagement = (
    df.groupby("IsActiveMember")["Balance"]
      .agg(["count", "mean", "median"])
      .reset_index()
)

balance_engagement

,IsActiveMember,count,mean,median
0,0,4849,77134.376863,98263.46
1,1,5151,75875.422145,96166.88


In [29]:
df.groupby("IsActiveMember")["Exited"].agg(
    Customers="count",
    Churned="sum",
    ChurnRate="mean"
).reset_index()

,IsActiveMember,Customers,Churned,ChurnRate
0,0,4849,1302,0.268509
1,1,5151,735,0.142691


In [30]:
# Create the High-Balance Disengaged Segment

df["HighBalanceDisengaged"] = (
    (df["IsActiveMember"] == 0) &
    (df["Balance"] > df["Balance"].median())
)

In [31]:
high_balance = df[df["HighBalanceDisengaged"]]

print("High-balance disengaged customers:", len(high_balance))
print("Churned:", high_balance["Exited"].sum())
print("Churn rate:", high_balance["Exited"].mean() * 100)

High-balance disengaged customers: 2456
Churned: 794
Churn rate: 32.328990228013026


In [32]:
# Credit Card Stickiness

card_churn = (
    df.groupby("HasCrCard")["Exited"]
      .agg(["count", "sum", "mean"])
      .reset_index()
)

card_churn["ChurnRate"] = card_churn["mean"] * 100

card_churn

,HasCrCard,count,sum,mean,ChurnRate
0,0,2945,613,0.208149,20.814941
1,1,7055,1424,0.201843,20.184266


In [33]:
df.groupby("HasCrCard")["Exited"].mean() * 100

HasCrCard
0    20.814941
1    20.184266
Name: Exited, dtype: float64

In [34]:
# Relationship Strength Index

df["RelationshipStrength"] = (
    df["IsActiveMember"] + df["NumOfProducts"]
)

df[["IsActiveMember", "NumOfProducts", "RelationshipStrength", "Exited"]].head()

,IsActiveMember,NumOfProducts,RelationshipStrength,Exited
0,1,1,2,1
1,1,1,2,0
2,0,3,3,1
3,0,2,2,0
4,1,1,2,0


In [35]:
# Churn by Relationship Strength

relationship_churn = (
    df.groupby("RelationshipStrength")["Exited"]
      .agg(["count", "sum", "mean"])
      .reset_index()
)

relationship_churn["ChurnRate"] = relationship_churn["mean"] * 100

relationship_churn

,RelationshipStrength,count,sum,mean,ChurnRate
0,1,2521,924,0.366521,36.652122
1,2,4707,697,0.148077,14.807733
2,3,2599,271,0.104271,10.427087
3,4,144,116,0.805556,80.555556
4,5,29,29,1.000000,100.000000


In [36]:
# Create retention categories

df["RelationshipTier"] = df["RelationshipStrength"].apply(
    lambda x: "Weak" if x == 1
    else "Moderate" if x in [2, 3]
    else "High-Risk"
)

df["RelationshipTier"].value_counts()

RelationshipTier
Moderate     7306
Weak         2521
High-Risk     173
Name: count, dtype: int64

In [37]:
# Final KPI summary

kpi_summary = {
    "Overall Churn Rate": df["Exited"].mean() * 100,
    "Active Customer Churn Rate": df[df["IsActiveMember"] == 1]["Exited"].mean() * 100,
    "Single-Product Churn Rate": df[df["NumOfProducts"] == 1]["Exited"].mean() * 100,
    "High-Balance Disengaged Churn Rate": df[df["HighBalanceDisengaged"]]["Exited"].mean() * 100
}

kpi_summary

{'Overall Churn Rate': np.float64(20.369999999999997),
 'Active Customer Churn Rate': np.float64(14.269073966220153),
 'Single-Product Churn Rate': np.float64(27.714398111723053),
 'High-Balance Disengaged Churn Rate': np.float64(32.328990228013026)}

In [39]:
# Save the final analytical dataset

df.to_csv("../data/customer_engagement_analysis.csv", index=False)

In [40]:
df.shape

(10000, 19)

In [ ]:
# Create the Streamlit file

